IMPORTAÇÃO DAS BIBLIOTECAS

In [1]:
# Biblioteca para manipulação dos dados
import pandas as pd

# Biblioteca para operações numéricas
import numpy as np

# Ferramentas de pré-processamento
from sklearn.preprocessing import StandardScaler

# Algoritmo K-Means
from sklearn.cluster import KMeans

CARREGAMENTO DO DATASET

In [2]:
df = pd.read_csv("dataset2_sem_duplicatas.csv")

df.head()

,id,fonte,texto,classe
0,1_0,sms,"Oi, peço para transferir aquele valor para meu...",Legitimate
1,2_0,sms,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate
2,3_0,sms,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate
3,4_0,sms,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate
4,5_0,sms,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate


INFORMAÇÕES BÁSICAS DO DATASET

In [3]:
# Mostra a quantidade de linhas e colunas
print("Formato do dataset:", df.shape)

# Mostra os nomes das colunas
print("\nColunas:")
print(df.columns.tolist())

# Mostra os tipos de dados
print("\nTipos das variáveis:")
print(df.dtypes)

Formato do dataset: (3397, 4)

Colunas:
['id', 'fonte', 'texto', 'classe']

Tipos das variáveis:
id        object
fonte     object
texto     object
classe    object
dtype: object


SEPARAÇÃO DAS VARIÁVEIS

In [6]:
X = df.drop(columns=["classe"])
y = df["classe"]

TRANSFORMAÇÃO DOS TEXTOS COM TF-IDF

In [8]:
# Importa a ferramenta responsável por transformar
# textos em representações numéricas
from sklearn.feature_extraction.text import TfidfVectorizer


# Seleciona somente a coluna de texto
X_texto = df["texto"]


# Cria o vetorizador TF-IDF
tfidf = TfidfVectorizer(

    # Converte todas as palavras para letras minúsculas
    lowercase=True,

    # Ignora palavras que aparecem em apenas 1 texto
    min_df=2,

    # Ignora palavras presentes em mais de 95% dos textos
    max_df=0.95,

    # Utiliza palavras isoladas e pares de palavras
    ngram_range=(1, 2),

    # Limita o número máximo de características
    max_features=10000
)


# Transforma os textos em números
X_tfidf = tfidf.fit_transform(X_texto)


# Mostra o formato da matriz criada
print("Formato após TF-IDF:", X_tfidf.shape)

Formato após TF-IDF: (3397, 10000)


CONFIGURAÇÃO DO K-MEANS

In [9]:
# Cria o modelo K-Means
kmeans = KMeans(

    # Serão formados dois grupos
    # Esperamos que eles representem padrões de
    # mensagens legítimas e mensagens de golpe
    n_clusters=2,

    # Estratégia utilizada para escolher
    # os centroides iniciais
    init="k-means++",

    # Quantidade de vezes que o algoritmo será
    # inicializado com centroides diferentes
    n_init=20,

    # Número máximo de iterações
    max_iter=300,

    # Tolerância utilizada para determinar
    # se o algoritmo convergiu
    tol=0.0001,

    # Algoritmo interno utilizado pelo K-Means
    algorithm="lloyd",

    # Mantém os resultados reproduzíveis
    random_state=42
)

TREINAMENTO DO K-MEANS

In [10]:
# Treina o modelo utilizando os textos
# já convertidos para valores numéricos pelo TF-IDF

kmeans.fit(X_tfidf)

print("Treinamento concluído com sucesso!")

Treinamento concluído com sucesso!


VISUALIZAÇÃO DOS CENTROIDES

In [11]:
# Cada centroide representa o "centro" de um cluster.
# Como os dados foram transformados com TF-IDF,
# cada posição corresponde a uma característica textual.

centroides = kmeans.cluster_centers_

print("Quantidade de clusters:", centroides.shape[0])
print("Quantidade de características:", centroides.shape[1])

Quantidade de clusters: 2
Quantidade de características: 10000


TERMOS MAIS REPRESENTATIVOS DE CADA CLUSTER

In [12]:
# Recupera os termos criados pelo TF-IDF
termos = tfidf.get_feature_names_out()

# Define quantos termos serão exibidos
quantidade_termos = 15

# Percorre os dois clusters
for numero_cluster in range(kmeans.n_clusters):

    print("\nCluster", numero_cluster)

    # Ordena os termos com maior peso no centroide
    indices = kmeans.cluster_centers_[
        numero_cluster
    ].argsort()[::-1]

    # Seleciona os termos mais representativos
    termos_importantes = [
        termos[i]
        for i in indices[:quantidade_termos]
    ]

    print(termos_importantes)


Cluster 0
['pesa', '00mt', 'saldo', 'facil', 'saldo pesa', 'pesa facil', 'confirmado', 'aos', 'de', 'pesa de', 'as', 'novo saldo', 'novo', 'teu', 'pm']

Cluster 1
['de', 'valor', 'para', 'que', 'em', 'conta', 'nome', 'com', 'você', 'um', 'manda', 'vem', 'no', 'eu', 'pesa']


ADICIONAR OS CLUSTERS AO DATASET

In [13]:
# Guarda no dataframe o cluster criado pelo K-Means
df["cluster"] = kmeans.labels_

# Mostra algumas linhas
df[
    ["id", "fonte", "texto", "classe", "cluster"]
].head(20)

,id,fonte,texto,classe,cluster
0,1_0,sms,"Oi, peço para transferir aquele valor para meu...",Legitimate,1
1,2_0,sms,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate,1
2,3_0,sms,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate,0
3,4_0,sms,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate,0
4,5_0,sms,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate,1
5,6_0,sms,estou a pedir aquele valor que estou a dever-te,Legitimate,1
6,7_0,sms,Millennium bim - Transferencia domestica - Val...,Legitimate,1
7,8_0,sms,Confirmado 7FA447X7O7C. Depositaste o valor de...,Legitimate,0
8,9_0,sms,"Confirmado 7HS24MB25EO. Transferiste 2,500.00M...",Legitimate,0
9,10_0,sms,"f**as, amanhã viajo. Podes passar daqui a casa...",Legitimate,1


COMPARAÇÃO ENTRE OS CLUSTERS E AS CLASSES REAIS

In [14]:
# Compara os clusters encontrados pelo K-Means
# com as classes reais existentes no dataset

comparacao = pd.crosstab(
    df["cluster"],
    df["classe"]
)

print(comparacao)

classe   Legitimate  Smishing
cluster                      
0               447         3
1              1559      1388


MAPEAMENTO DOS CLUSTERS PARA AS CLASSES

In [15]:
# Cria um dicionário que associa cada cluster
# à classe que aparece com maior frequência nele

mapa_clusters = {}

for cluster in df["cluster"].unique():

    # Seleciona as classes reais dos registros
    # pertencentes ao cluster atual
    classes_cluster = df.loc[
        df["cluster"] == cluster,
        "classe"
    ]

    # Identifica a classe mais frequente
    classe_majoritaria = classes_cluster.mode()[0]

    # Salva a correspondência
    mapa_clusters[cluster] = classe_majoritaria


print("Mapeamento dos clusters:")
print(mapa_clusters)

Mapeamento dos clusters:
{np.int32(1): 'Legitimate', np.int32(0): 'Legitimate'}


CRIAÇÃO DA CLASSE PREVISTA

In [16]:
# Converte o número do cluster para o nome
# da classe correspondente

df["classe_prevista"] = df["cluster"].map(
    mapa_clusters
)

# Visualiza o resultado
df[
    ["texto", "classe", "cluster", "classe_prevista"]
].head(20)

,texto,classe,cluster,classe_prevista
0,"Oi, peço para transferir aquele valor para meu...",Legitimate,1,Legitimate
1,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate,1,Legitimate
2,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate,0,Legitimate
3,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate,0,Legitimate
4,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate,1,Legitimate
5,estou a pedir aquele valor que estou a dever-te,Legitimate,1,Legitimate
6,Millennium bim - Transferencia domestica - Val...,Legitimate,1,Legitimate
7,Confirmado 7FA447X7O7C. Depositaste o valor de...,Legitimate,0,Legitimate
8,"Confirmado 7HS24MB25EO. Transferiste 2,500.00M...",Legitimate,0,Legitimate
9,"f**as, amanhã viajo. Podes passar daqui a casa...",Legitimate,1,Legitimate
